## 증권 주요뉴스 수집

### HTTP 요청 실행

In [ ]:
# 필요한 모듈을 임포트합니다.
import requests

In [ ]:
# 요청 URL을 설정합니다.
url = 'https://finance.naver.com/news/mainnews.naver'

In [ ]:
# 쿼리 문자열을 딕셔너리로 생성합니다.
# [참고] Payload에서 복사한 값을 딕셔너리로 생성합니다.
query = dict(date='2026-01-01', page=1)

In [ ]:
# HTTP 요청을 실행하고 res에 할당합니다.
# [참고] params 매개변수에 쿼리 문자열을 지정합니다.
res = requests.get(url=url, params=query)

### HTTP 응답 확인

In [ ]:
# HTTP 응답 상태 코드를 확인합니다.
res.status_code

In [ ]:
# HTTP 응답 헤더에서 콘텐트 형태를 확인합니다.
res.headers['content-type']

In [ ]:
# HTTP 응답 바디 문자열을 출력합니다.
print(res.text)

In [ ]:
# HTTP 요청 URL을 출력합니다.
print(res.url)

### HTML 데이터 처리

In [ ]:
# 필요한 모듈을 임포트합니다.
from bs4 import BeautifulSoup as BTS

In [ ]:
# HTTP 응답 바디 문자열을 bs4.BeautifulSoup으로 변환하고 soup에 할당합니다.
soup = BTS(markup=res.text, features='html.parser')

In [ ]:
# soup을 확인합니다.
soup

In [ ]:
# soup을 HTML의 계층 구조(들여쓰기)를 반영한 문자열로 변환하여 출력합니다.
print(soup.prettify())

### 뉴스 정보 수집

In [ ]:
# 뉴스 제목을 포함하는 HTML 요소를 모두 선택하고 items에 할당합니다.
# [참고] 썸네일 유무에 따라 <dt>와 <dd> 태그로 달라집니다.
items = soup.select(selector='.articleSubject > a')

In [ ]:
# items의 원소 개수를 확인합니다.
len(items)

In [ ]:
# items의 첫 번째 원소에서 뉴스 제목을 반환합니다.
items[0].text

In [ ]:
# items의 첫 번째 원소에서 뉴스 링크를 반환합니다.
items[0]['href']

### 기존 클래스를 상속하고 메서드 추가

In [ ]:
# bs4.BeautifulSoup 클래스를 상속하고, 링크를 리스트로 반환하는 메서드를 추가합니다.
class MyBTS(BTS):
    def html_text(self, css_selector):
        items = self.select(selector=css_selector)
        return [item.text for item in items]
    def html_attr(self, css_selector, key='href'):
        items = self.select(selector=css_selector)
        return [item[key] for item in items]

### 뉴스 정보 수집

In [ ]:
# MyBTS 클래스로 soup2를 생성합니다.
soup2 = MyBTS(markup=res.text, features='html.parser')

In [ ]:
# 뉴스 제목을 리스트로 반환합니다.
soup2.html_text(css_selector='.articleSubject > a')

In [ ]:
# 뉴스 링크를 리스트로 반환합니다.
soup2.html_attr(css_selector='.articleSubject > a')

In [ ]:
# 뉴스 등록일을 리스트로 반환합니다.
soup2.html_text(css_selector='.articleSummary > span.wdate')

### 데이터프레임 생성

In [ ]:
# 필요한 모듈을 임포트합니다.
import pandas as pd

In [ ]:
# 제목, 링크 및 등록일 열을 갖는 데이터프레임을 생성합니다.
pd.DataFrame(data={
    '제목': soup2.html_text(css_selector='.articleSubject > a'), 
    '링크': soup2.html_attr(css_selector='.articleSubject > a'), 
    '등록일': soup2.html_text(css_selector='.articleSummary > span.wdate')
})

## End of Document